In [0]:
# Package install for quickstart
!pip install ibm-aigov-facts-client
!pip install python-dotenv

In [0]:
import os
from dotenv import load_dotenv
import requests, json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from keras.models import Sequential
from keras.layers import LSTM, Dense

from ibm_aigov_facts_client import AIGovFactsClient, CloudPakforDataConfig, DetachedPromptTemplate, PromptTemplate, DeploymentDetails, ModelDetails


# Track ML Model

Initialize fact sheets client

In [0]:
load_dotenv(".env")
CPD_URL = os.getenv("CPD_URL")
CPD_USERNAME = os.getenv("CPD_USERNAME")
CPD_PASSWORD = os.getenv("CPD_PASSWORD")

In [0]:
creds=CloudPakforDataConfig(
    service_url=CPD_URL,
    username=CPD_USERNAME,
    password=CPD_PASSWORD
)

# Generate a unique experiment name
# unique_suffix = str(uuid.uuid4())[:8]
experiment_name = "amazon-stock-price-0919"

# container_type= "space"
# deployment_space_id= "9815180d-01fc-4f56-87ce-dc9e907f36fd" #TD ML development Space


facts_client = AIGovFactsClient(
    cloud_pak_for_data_configs=creds,
    experiment_name=experiment_name,
    # container_type= container_type,
    # container_id= deployment_space_id,
    external_model=True,
    enable_autolog=True,
    set_as_current_experiment=True
)

## Prepare training data

In [0]:
# Load sample Amazon stock CSV from local
# df = pd.read_csv("/kaggle/input/amazonstockprice/AMZN_Stock_Updated_V2.csv")
df = spark.read.format("delta").load("dbfs:/user/hive/warehouse/amzn_stock_updated_v_2")
df = df.toPandas()
# df.drop(columns=["Unnamed: 0"], inplace=True)
df["Date"] = pd.to_datetime(df["Date"])
df.set_index("Date", inplace=True)
data = df["High"]

# Prepare training data
train_data = data.iloc[:-4]
X_train, y_train = [], []
for i in range(2, len(train_data)):
    X_train.append(train_data[i-2:i])
    y_train.append(train_data[i])
X_train, y_train = np.array(X_train), np.array(y_train)
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))

## Train model

In [0]:
# LSTM Model
model = Sequential()
model.add(LSTM(50, activation='relu', input_shape=(X_train.shape[1], 1)))
model.add(Dense(25))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mean_squared_error')
model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=2)

In [0]:
model.save(f"model/{experiment_name}.keras")

## Evaluate model

In [0]:
from sklearn.metrics import mean_squared_error
import numpy as np

# Evaluate model on test data
test_data = train_data[-(len(y_train)):]
X_val, Y_val = [], []
for i in range(2, len(test_data)):
    X_val.append(test_data[i-2:i])
    Y_val.append(test_data[i])
X_val, Y_val = np.array(X_val), np.array(Y_val)
X_val = X_val.reshape((X_val.shape[0], X_val.shape[1], 1))
predictions = model.predict(X_val)

rmse_train = float(np.sqrt(mean_squared_error(y_train, model.predict(X_train))))
rmse_val = float(np.sqrt(mean_squared_error(Y_val, predictions)))

print("Train RMSE:", rmse_train)
print("Validation RMSE:", rmse_val)


In [0]:
training_data_reference = {
    "id": "amzn_stock_updated_v_2",            
    "type": "fs",                             
    "location": {
        "path": "dbfs:/user/hive/warehouse/amzn_stock_updated_v_2",
        "connection": {
            "type": "dbfs"
        }
    },
    "data_type": "csv",
    "description": "Training data stored in DBFS for LSTM model"
}
model_stub_details = {
    "frameworkName": "TensorFlow",
    "frameworkVersion": "2.4.1",
    "algorithm": "LSTM",
    "epochs": 10,
    "rmse_train": rmse_train,
    "rmse_val": rmse_val
}


## Export facts, save model asset

In [0]:
current_experiment_id= facts_client.experiments.get_current_experiment_id()
facts_client.runs.list_runs_by_experiment(current_experiment_id)

In [0]:
# paste in the current run ID
current_run_ID= "4b5fd743b65d4f689f666805456aae31"
facts_client.export_facts.export_payload(current_run_ID)

## Save model to model inventory

In [0]:
inventory_id="e24ff319-4ee8-4716-a346-bb6e23383167" # TD Model Inventory

external_model=facts_client.external_model_facts.save_external_model_asset(model_identifier= experiment_name,
                                                                           name=experiment_name,
                                                                           model_details=model_stub_details,
                                                                           training_data_reference=training_data_reference,
                                                                           description="Model developed in Azure databricks to predict Amazon Stock prices",
                                                                           catalog_id=inventory_id)

## Track model to AI Use Case

In [0]:
usecase_id="ff4b8f6a-0b66-4265-b891-024e4008860f" # Stock Prediction Use Case
approach_id= "00000000-0000-0000-0000-000000000000" # default approach

# get AI use case
ai_usecase=facts_client.assets.get_ai_usecase(catalog_id=inventory_id,
                                              ai_usecase_id=usecase_id)
#get default approach
approach=ai_usecase.get_approach(approach_id= approach_id)

external_model.track(usecase=ai_usecase,approach=approach, version_number="0.0.8",version_comment="external model minor version")


## Push Custom Metrics

In [0]:
# model= facts_client.assets.get_model(model_id=model_id, container_id= inventory_id)
run= model.get_experiment().get_run()

In [0]:
metrics = {"prod_RMSE": 0.65}
run.set_custom_run_facts(metrics=metrics, overwrite=False, debug=False)